## Model Training

This sections serves to use the prepared and cleaned dataset and use the model to train with the dataset.

### Dataset Loading

In [1]:
## Load Dataset

import pandas as pd
import numpy as np
from dataset_class.job_post_dataset import JobPostingDataset
import torch
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

combined_df = pd.read_csv("./data/clean/fake_job_postings_ALL.csv")


numeric_cols = ["telecommuting", "missing_count", "total_text_len", "company_profile_len", "description_len", 
                "requirements_len", "company_profile_word_count", "description_word_count", 
                "requirements_word_count", "salary_provided", "has_company_profile",
                "vague_location", "has_company_logo", "has_questions","benefits_len","benefits_word_count",]
non_binary_cols = [
    "missing_count",          
    "total_text_len", 
    "company_profile_len", 
    "description_len", 
    "requirements_len", 
    "benefits_len", 
    "company_profile_word_count", 
    "description_word_count", 
    "requirements_word_count", 
    "benefits_word_count"
]
numerical_array = combined_df[numeric_cols].to_numpy(dtype=np.float32)  # (N, 16)
labels_array    = combined_df['fraudulent'].to_numpy(dtype=np.float32)  # (N,)

print(numerical_array.shape)
print(labels_array.shape)



(17880, 16)
(17880,)


### Train - Test - Validation Split

Conducted a 80 - 10 - 10 split for train, test and validation, stratifying with the label column to ensure the same distribution in each dataset.

In [2]:
from sklearn.model_selection import train_test_split

# Split 80% Train, 20% "Rest" (temp_data)
# Stratify based on 'fraudulent' column to maintain class distribution in both sets
train_data, temp_data = train_test_split(
    combined_df, test_size=0.2, random_state=42, stratify=combined_df['fraudulent']
)

# Split that 20% into half (10% Val, 10% Test)
# Use temp_data['fraudulent'] for stratification
val_data, test_data = train_test_split(
    temp_data, test_size=0.5, random_state=42, stratify=temp_data['fraudulent']
)

# Convert to lists and numpy arrays to store as tensors later

X_train_text = train_data['full_text'].tolist()
X_train_numeric = train_data[numeric_cols].values.astype(np.float32)
y_train = train_data['fraudulent'].values.tolist()

X_val_text = val_data['full_text'].tolist()
X_val_numeric = val_data[numeric_cols].values.astype(np.float32)
y_val = val_data['fraudulent'].values.tolist()

X_test_text = test_data['full_text'].tolist()
X_test_numeric = test_data[numeric_cols].values.astype(np.float32)
y_test = test_data['fraudulent'].values.tolist()

### Non-binary value Standardisation

We standardise only the non-binary numeric values 

In [3]:
from sklearn.preprocessing import StandardScaler

non_binary_cols = [
    "missing_count",          
    "total_text_len", 
    "company_profile_len", 
    "description_len", 
    "requirements_len", 
    "benefits_len", 
    "company_profile_word_count", 
    "description_word_count", 
    "requirements_word_count", 
    "benefits_word_count"
]

non_binary_indices = [numeric_cols.index(col) for col in non_binary_cols]

scaler = StandardScaler()

# Scale in-place, preserving original column order
X_train_numeric[:, non_binary_indices] = scaler.fit_transform(X_train_numeric[:, non_binary_indices])
X_val_numeric[:,   non_binary_indices] = scaler.transform(X_val_numeric[:,     non_binary_indices])
X_test_numeric[:,  non_binary_indices] = scaler.transform(X_test_numeric[:,    non_binary_indices])

# Verify
print(X_train_numeric.mean(axis=0).round(3))  # non-binary cols ≈ 0.0
print(X_train_numeric.std(axis=0).round(3))   # non-binary cols ≈ 1.0

[ 0.042 -0.     0.    -0.     0.     0.     0.     0.    -0.     0.16
  0.813  0.025  0.793  0.493  0.    -0.   ]
[0.2   1.    1.    1.    1.    1.    1.    1.    1.    0.366 0.39  0.156
 0.405 0.5   1.    1.   ]


In [4]:
# Verify all types before creating datasets
print(type(X_train_text),    type(X_train_text[0]))     # list, str
print(type(X_train_numeric), X_train_numeric.shape)     # ndarray, (N, 16)
print(type(y_train),         type(y_train[0]))          # list, int

<class 'list'> <class 'str'>
<class 'numpy.ndarray'> (14304, 16)
<class 'list'> <class 'int'>


In [5]:
print(X_train_numeric.mean(axis=0))   # should be close to 0
print(X_train_numeric.std(axis=0))    # should be close to 1

[ 4.1596755e-02 -1.2800998e-08  5.0670614e-09 -6.9338735e-09
  6.4004988e-09  1.0667498e-09  2.1468340e-08  3.7336241e-09
 -3.7336241e-09  1.5960571e-01  8.1319910e-01  2.4958054e-02
  7.9299498e-01  4.9293903e-01  2.4001871e-08 -7.8672802e-09]
[0.19966587 0.99999994 1.         1.         1.         1.
 1.         1.         1.         0.36623996 0.3897516  0.15599728
 0.40515912 0.4999501  1.         1.        ]


### Loading of fine-tuned FastText model

In [6]:
from gensim.models import FastText

fasttext_model_optimal = FastText.load("./optimal_fasttext.bin")

print(fasttext_model_optimal.wv.similarity('software', 'engineer'))
print(fasttext_model_optimal.wv.similarity('skills', 'experience'))
print(fasttext_model_optimal.wv.most_similar('water', topn=10))

C:\Users\HP Victus\AppData\Roaming\Python\Python313\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


0.43055043
0.51230246
[('saltwater', 0.8357968330383301), ('wastewater', 0.8045268654823303), ('backwater', 0.7969263195991516), ('stormwater', 0.7853497266769409), ('watering', 0.7785604596138), ('groundwater', 0.7723948955535889), ('whitewater', 0.7641539573669434), ('stillwater', 0.7530474662780762), ('waterloo', 0.729570746421814), ('deepwater', 0.7263585329055786)]


### Tokenizing strings in each sample

In [7]:
from nltk.tokenize import word_tokenize

train_tk_sentences = [word_tokenize(text.lower()) for text in X_train_text]  
val_tk_sentences = [word_tokenize(text.lower()) for text in X_val_text]  
test_tk_sentences = [word_tokenize(text.lower()) for text in X_test_text]  

In [8]:
print(train_tk_sentences[0])

['contact', 'center', 'representatives', 'us', 'va', 'virginia', 'beach', 'tidewater', 'finance', 'co.', 'was', 'established', 'in', '1992', 'for', 'the', 'initial', 'purpose', 'of', 'purchasing', ',', 'and', 'servicing', 'retail', 'installment', 'contracts', '.', 'there', 'are', 'two', 'divisions', 'tidewater', 'credit', 'services', ',', 'providing', 'indirect', 'consumer', 'retail', 'finance', 'options', 'and', 'tidewater', 'motor', 'credit', ',', 'providing', 'indirect', 'consumer', 'auto', 'financing', '.', 'we', 'remain', 'committed', 'to', 'offering', 'a', 'partnership', 'with', 'the', 'dealers', 'and', 'consumers', 'to', 'create', 'a', 'win', 'win', 'win', 'situation', '.', 'our', 'success', 'relies', 'solely', 'on', 'the', 'success', 'of', 'our', 'dealers', 'and', 'our', 'consumers', '.', 'full', 'time', 'positions', 'include', 'the', 'following', 'benefits', '40', 'vacation', 'hours', 'after', '6', 'months', 'of', 'employment', ',', '80', 'vacation', 'hours', 'after', '1', 'ye

### Encoding tokens into numerical value

In [9]:
# Reserve 0 = <unk>, 1 = <pad>
vocab = {"<unk>": 0, "<pad>": 1}

# Start FastText words from index 2 onwards
for idx, word in enumerate(fasttext_model_optimal.wv.index_to_key):
    vocab[word] = idx + 2

print(f"Vocab size: {len(vocab)}")
print(f"<unk> index: {vocab['<unk>']}")  # 0
print(f"<pad> index: {vocab['<pad>']}")  # 1

def encode(tokenized_sentence, vocab):
    return [vocab.get(token, 0) for token in tokenized_sentence]  # 0 = <unk>

train_texts_tok = [encode(s, vocab) for s in train_tk_sentences]
val_texts_tok   = [encode(s, vocab) for s in val_tk_sentences]
test_texts_tok  = [encode(s, vocab) for s in test_tk_sentences]

Vocab size: 22096
<unk> index: 0
<pad> index: 1


### Instantiating Datasets

In [10]:
from torch.utils.data import DataLoader

train_dataset = JobPostingDataset(
    texts_tok           = train_texts_tok,  # your tokenized sequences
    numerical_features  = X_train_numeric,  # your scaled numerical features
    labels              = y_train,          # your labels
    max_len             = 2048              # your max sequence length (for padding/truncation)
)
train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True) # Set shuffle=True for training

In [11]:
val_dataset = JobPostingDataset(
    texts_tok           = val_texts_tok,  
    numerical_features  = X_val_numeric,
    labels              = y_val,
    max_len             = 2048
)
val_dataloader = DataLoader(val_dataset, batch_size=64, shuffle=False)

In [12]:
test_dataset = JobPostingDataset(
    texts_tok           = test_texts_tok,  
    numerical_features  = X_test_numeric,
    labels              = y_test,
    max_len             = 2048
)
test_dataloader = DataLoader(test_dataset, batch_size=64, shuffle=False)

### Testing Dataloader by sampling a batch

In [13]:
inputs,label = train_dataset[0]
print("input_ids shape:          ", inputs["input_ids"].shape)           # (2048,)
print("attention_mask shape:     ", inputs["attention_mask"].shape)      # (2048,)
print("numerical_features shape: ", inputs["numerical_features"].shape)  # (num_features,)
print("label:                    ", label)


input_ids shape:           torch.Size([2048])
attention_mask shape:      torch.Size([2048])
numerical_features shape:  torch.Size([16])
label:                     tensor(0.)


### Creating the embedding matrix

This is used to copy into the embedding layer


In [14]:
import torch

vocab_size = len(vocab)
embed_dim  = fasttext_model_optimal.wv.vector_size

embedding_matrix = np.zeros((vocab_size, embed_dim), dtype=np.float32)
# index 0 (<unk>) stays zero
# index 1 (<pad>) stays zero

for word, idx in vocab.items():
    if word in fasttext_model_optimal.wv:
        embedding_matrix[idx] = fasttext_model_optimal.wv[word]

pretrained_embeddings = torch.tensor(embedding_matrix, dtype=torch.float)
print(pretrained_embeddings.shape)  # torch.Size([vocab_size, 100])

torch.Size([22096, 100])


### Model Instantiation

In [15]:
from model_construction.model import FakeJobDetector

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = FakeJobDetector(
    vocab_size=vocab_size,
    embed_dim=embed_dim,
    num_numerical_features=X_train_numeric.shape[1],
    pretrained_embeddings=pretrained_embeddings,
    gru_hidden_dim=64,
    num_hidden_dim=128,
    device=device,
)

print(model)

FakeJobDetector(
  (embedding): Embedding(22096, 100, padding_idx=1)
  (bigru): BiGRUBlock(
    (gru): GRU(100, 64, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  )
  (attention): MultiHeadAttentionPooling(
    (heads): ModuleList(
      (0-1): 2 x Linear(in_features=128, out_features=1, bias=True)
    )
    (projection): Linear(in_features=256, out_features=128, bias=True)
    (dropout): Dropout(p=0.3, inplace=False)
  )
  (numerical): NumericalBlock(
    (layers): Sequential(
      (0): Linear(in_features=16, out_features=128, bias=True)
      (1): ReLU()
      (2): Dropout(p=0.3, inplace=False)
      (3): Linear(in_features=128, out_features=128, bias=True)
      (4): ReLU()
      (5): Dropout(p=0.3, inplace=False)
    )
  )
  (classifier): Linear(in_features=256, out_features=1, bias=True)
)


### Model Training

Training with a chosen number of epochs and learning rate and a dedicated saving path to store the best weights.

In [16]:
train_losses, val_losses = model.fit(
    dataloader     = train_dataloader,
    val_dataloader = val_dataloader,
    num_epochs     = 20,
    learning_rate  = 1e-3,
    save_path      = "best_model.pt",
)





Epoch 1/20 | Train Loss: 0.0457 | Val Loss: 0.0411
  ✅ Best model saved (val_loss=0.0411)
Epoch 2/20 | Train Loss: 0.0384 | Val Loss: 0.0373
  ✅ Best model saved (val_loss=0.0373)
Epoch 3/20 | Train Loss: 0.0330 | Val Loss: 0.0276
  ✅ Best model saved (val_loss=0.0276)
Epoch 4/20 | Train Loss: 0.0262 | Val Loss: 0.0244
  ✅ Best model saved (val_loss=0.0244)
Epoch 5/20 | Train Loss: 0.0205 | Val Loss: 0.0184
  ✅ Best model saved (val_loss=0.0184)
Epoch 6/20 | Train Loss: 0.0176 | Val Loss: 0.0206
  ⚠️ No improvement (best_val_loss=0.0184)
Epoch 7/20 | Train Loss: 0.0177 | Val Loss: 0.0158
  ✅ Best model saved (val_loss=0.0158)
Epoch 8/20 | Train Loss: 0.0165 | Val Loss: 0.0145
  ✅ Best model saved (val_loss=0.0145)
Epoch 9/20 | Train Loss: 0.0149 | Val Loss: 0.0150
  ⚠️ No improvement (best_val_loss=0.0145)
Epoch 10/20 | Train Loss: 0.0146 | Val Loss: 0.0212
  ⚠️ No improvement (best_val_loss=0.0145)
Epoch 11/20 | Train Loss: 0.0167 | Val Loss: 0.0167
  ⚠️ No improvement (best_val_loss=

### Model Evaluation
This is done with the test dataset.

In [17]:
model.evaluate(test_dataloader, threshold=0.5)

              precision    recall  f1-score   support

        Real       0.99      1.00      0.99      1702
        Fake       0.93      0.86      0.89        86

    accuracy                           0.99      1788
   macro avg       0.96      0.93      0.94      1788
weighted avg       0.99      0.99      0.99      1788



### Model Branch Evaluation

Compare the performance of both the NLP and numeric branch to observe the contributions made to the final scores in the classification matrix

In [18]:
model.evaluate_branches(test_dataloader, threshold=0.3)


Full model
              precision    recall  f1-score   support

        Real       1.00      0.97      0.98      1702
        Fake       0.62      0.91      0.74        86

    accuracy                           0.97      1788
   macro avg       0.81      0.94      0.86      1788
weighted avg       0.98      0.97      0.97      1788


NLP only
              precision    recall  f1-score   support

        Real       1.00      0.97      0.98      1702
        Fake       0.62      0.92      0.74        86

    accuracy                           0.97      1788
   macro avg       0.81      0.95      0.86      1788
weighted avg       0.98      0.97      0.97      1788


Numerical only
              precision    recall  f1-score   support

        Real       0.97      0.77      0.86      1702
        Fake       0.11      0.57      0.19        86

    accuracy                           0.77      1788
   macro avg       0.54      0.67      0.53      1788
weighted avg       0.93      0.77   

C:\Users\HP Victus\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\HP Victus\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\HP Victus\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{me

### Hyperparameter tuning of threshold

In [19]:
for threshold in [0.3, 0.4, 0.5, 0.6, 0.7]:
    print(f"\n--- Threshold: {threshold} ---")
    model.evaluate(test_dataloader, threshold=threshold)


--- Threshold: 0.3 ---
              precision    recall  f1-score   support

        Real       1.00      0.97      0.98      1702
        Fake       0.62      0.91      0.74        86

    accuracy                           0.97      1788
   macro avg       0.81      0.94      0.86      1788
weighted avg       0.98      0.97      0.97      1788


--- Threshold: 0.4 ---
              precision    recall  f1-score   support

        Real       0.99      0.99      0.99      1702
        Fake       0.86      0.88      0.87        86

    accuracy                           0.99      1788
   macro avg       0.93      0.94      0.93      1788
weighted avg       0.99      0.99      0.99      1788


--- Threshold: 0.5 ---
              precision    recall  f1-score   support

        Real       0.99      1.00      0.99      1702
        Fake       0.93      0.86      0.89        86

    accuracy                           0.99      1788
   macro avg       0.96      0.93      0.94      1788
we